# Differential expression → view

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmdcolin/jbrowse-anywidget/blob/main/examples/07_differential_expression.ipynb)

Another analysis→genome loop: run a small DE analysis over gene counts, then load each gene colored by its result — up-regulated red, down-regulated blue. All computed in the notebook (numpy only).

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/cmdcolin/jbrowse-anywidget" pandas numpy

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## Counts → log2 fold-change and a t-test

Simulate control vs treatment counts for a panel of genes (a few truly differential), then compute per-gene log2FC and a Welch t-test — a normal approximation gives the p-value, no scipy needed. Swap in your own DESeq2/edgeR table joined to gene coordinates.

In [ ]:
import math
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n_genes, n_rep = 80, 4
chrom, gene_len, gap = "7", 6_000, 40_000
starts = 1_000_000 + np.arange(n_genes) * gap

base = rng.uniform(20, 400, n_genes)  # baseline expression per gene
true_lfc = np.zeros(n_genes)
up = rng.choice(n_genes, 6, replace=False)
down = rng.choice(np.setdiff1d(np.arange(n_genes), up), 6, replace=False)
true_lfc[up] = rng.uniform(1.5, 3.0, up.size)
true_lfc[down] = -rng.uniform(1.5, 3.0, down.size)

ctrl = rng.poisson(base[:, None], size=(n_genes, n_rep))
treat = rng.poisson((base * 2.0**true_lfc)[:, None], size=(n_genes, n_rep))

lc, lt = np.log2(ctrl + 1), np.log2(treat + 1)
lfc = lt.mean(1) - lc.mean(1)
se = np.sqrt(lc.var(1, ddof=1) / n_rep + lt.var(1, ddof=1) / n_rep)
t = np.divide(lfc, se, out=np.zeros_like(lfc), where=se > 0)
pval = np.array([math.erfc(abs(v) / math.sqrt(2)) for v in t])

de = pd.DataFrame(
    {
        "chrom": chrom,
        "start": starts,
        "end": starts + gene_len,
        "name": [f"GENE{i:04d}" for i in range(n_genes)],
        "log2fc": lfc.round(2),
        "pvalue": pval,
    }
)
de["sig"] = np.where(
    (de.pvalue < 0.01) & (de.log2fc.abs() > 1),
    np.where(de.log2fc > 0, "up", "down"),
    "ns",
)
de.sort_values("pvalue").head()

## Load the DE table onto the genome

Each gene is colored by call; `log2fc`/`pvalue` ride along and show in the feature details.

In [ ]:
from jbrowse_anywidget import LinearGenomeView, make_assembly

grch38 = make_assembly(
    "GRCh38",
    "https://s3.amazonaws.com/jbrowse.org/genomes/GRCh38/fasta/GRCh38.fa.gz",
    aliases=["hg38"],
)
view = LinearGenomeView(assembly=grch38, location="7:1,000,000..4,300,000")
view.add_features(
    de,
    name="differential expression",
    color="jexl:get(feature,'sig') == 'up' ? '#c62828' : get(feature,'sig') == 'down' ? '#1565c0' : '#cfcfcf'",
)
view